In [ ]:
!pip install sentence-transformers umap-learn
from google.colab import files
uploaded_files = files.upload() # upload the local files from the computer

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import sys
from sentence_transformers import SentenceTransformer   # the SBERT model
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import umap
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

In [ ]:
sys.path.insert(0,'/content') # look in /content for any local files we uploaded

In [ ]:
df = pd.read_csv('/content/data_preprocessed_csv.csv')

df['cleaned_abstract'] = df['cleaned_abstract'].fillna('') # handling missing values
df['abstract'] = df['abstract'].fillna('')

print(f"Total number of papers : {len(df)}")
print(f"Number of columns : {df.columns.tolist()}")

In [ ]:
model_used = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Extracts abstracts and encode them into numerical embeddings using the model
abstracts = df['abstract'].tolist()
embeddings = model_used.encode(abstracts,batch_size=64,convert_to_numpy=True)
print(f"Embeddings shape : {embeddings.shape}")

In [ ]:
np.save('/content/embeddings.npy',embeddings) # save the embeddings to the computer
files.download('/content/embeddings.npy')

In [ ]:
similarity_score = cosine_similarity(embeddings[0:1],embeddings[1:2])[0][0] # Picking 2 random papers and checking their similarity score
print("\nPaper 1 is about :",df['abstract'].iloc[0][:400])
print("\nPaper 2 is about :",df['abstract'].iloc[1][:400])
print(f"Their similarity score is : {similarity_score : }") # Range betwen 0 to 1, higher means more similar

In [ ]:
def table_of_similar_papers(query, top_k=10):
    query_embedding = model_used.encode([query],convert_to_numpy=True) #turning our search query into 384 numbers
    scores = cosine_similarity(query_embedding,embeddings)[0] # comparing query against every other paper's embedding
    top_results = scores.argsort()[::-1][:top_k] # grabbing top results after sorting
    results = df.iloc[top_results][['title','year','abstract']].copy()
    results['similarity score'] = scores[top_results].round(4)
    results['abstract'] = results['abstract'].str[:300]
    return results.reset_index(drop=True)

In [ ]:
table_of_similar_papers("deep learning")

In [ ]:
table_of_similar_papers("Natural language processing")

In [ ]:
number_of_clusters = 10
kmeans_clustering = KMeans(n_clusters=number_of_clusters,random_state=33)
df['kmeans_clusters'] = kmeans_clustering.fit_predict(embeddings)
print("Number of papers in each cluster : \n")
print(df['kmeans_clusters'].value_counts().sort_index())

In [ ]:
print("Few paper titles from each cluster :\n")
for c in range(number_of_clusters):
    paper_titles = df[df['kmeans_clusters'] == c]['title'].dropna().head(10).tolist()
    print(f"Few Papers in the Cluster {c}:")
    for title in  paper_titles:
        print(f"--> {title}")
    print()

In [ ]:
reducer_umap = umap.UMAP(n_components=2,random_state=33) #fitting umap

coordinates = reducer_umap.fit_transform(embeddings) # fit transform

# To store the x and y coordinates in the table
df['umap_x_coordn'] = coordinates[:,0]   # 1st column
df['umap_y_coordn'] = coordinates[:,1]   # 2nd column

fig, ax = plt.subplots(figsize=(15,7))

scatter = ax.scatter(df['umap_x_coordn'],df['umap_y_coordn'],c=df['kmeans_clusters'],cmap='tab10',s=2,alpha=0.5)

plt.colorbar(scatter,ax=ax,label='Clusters')

ax.set_title('UMAP of SBERT embeddings',fontsize=12)
ax.set_xlabel('UMAP dimension 1')
ax.set_ylabel('UMAP dimension 2')
plt.tight_layout()

reducer_umap = umap.UMAP(n_components=2,random_state=33) #fitting umap

coordinates = reducer_umap.fit_transform(embeddings) # fit transform

# To store the x and y coordinates in the table
df['umap_x_coordn'] = coordinates[:,0]   # 1st column
df['umap_y_coordn'] = coordinates[:,1]   # 2nd column

fig, ax = plt.subplots(figsize=(15,7))

scatter = ax.scatter(df['umap_x_coordn'],df['umap_y_coordn'],c=df['kmeans_clusters'],cmap='tab10',s=2,alpha=0.5)

plt.colorbar(scatter,ax=ax,label='Clusters')

ax.set_title('UMAP of SBERT embeddings',fontsize=12)
ax.set_xlabel('UMAP - 1st dimension')
ax.set_ylabel('UMAP - 2nd dimension')
plt.tight_layout()


In [ ]:
cluster_labels = {0:'Social Media and Emotions',1:'Dialogue and Summarisation',2:'AI Foundations',3:'Documentation and Language Standards',4:'Reasoning Under Uncertainty',5:'Speech and Translation',6:'Classic Machine Learning',7:'Agents and Decision Making',8:'Math Behind Model Training',9:'Finding and Linking Information'}

for cluster_id, label in cluster_labels.items(): # Label positioning
    mask = df['kmeans_clusters'] == cluster_id
    centre_x = df.loc[mask,'umap_x_coordn'].mean()
    centre_y = df.loc[mask,'umap_y_coordn'].mean()
    ax.annotate(label,(centre_x, centre_y),fontsize=9,fontweight='bold',ha='center',bbox=dict(boxstyle='round,pad=0.2',fc='white',alpha=0.7,ec='none'))

In [ ]:
plt.savefig('/content/umap_clusters_nithya_sbert.png',dpi=150)
plt.show()

files.download('/content/umap_clusters_nithya_sbert.png')

In [ ]:
print(df.head())

In [ ]:
print(df['period'].value_counts().sort_index())

In [ ]:
print("Oldest Year :",df['year'].min())
print("Latest Year :",df['year'].max())
print("Total num of papers :",len(df))

In [ ]:
model_specter = SentenceTransformer('sentence-transformers/allenai-specter')  #using specter model to compare with MiniLM 

In [ ]:
similarity_score_comparision = model_specter.encode([df['abstract'].iloc[0], df['abstract'].iloc[1]],convert_to_numpy=True)
specter_similarity_score = cosine_similarity(similarity_score_comparision[0:1],similarity_score_comparision[1:2])[0][0]
print(f"MiniLM similarity score : {similarity_score :}")
print(f"Specter similarity score : {specter_similarity_score :}")

In [ ]:
specter_model_embeddings = model_specter.encode(df['abstract'].tolist(),batch_size=64,convert_to_numpy=True,show_progress_bar=True)
print(specter_model_embeddings.shape)

In [ ]:
np.save('/content/specter_model_embeddings.npy',specter_model_embeddings)
files.download('/content/specter_model_embeddings.npy')

In [ ]:
k_means_test = KMeans(n_clusters=10,random_state=33)

minilm_label  = k_means_test.fit_predict(embeddings)
minilm_similarity_score = silhouette_score(embeddings,minilm_label,metric='cosine',sample_size=3000)

specter_label = k_means_test.fit_predict(specter_model_embeddings)
specter_similarity_score = silhouette_score(specter_model_embeddings,specter_label,metric='cosine',sample_size=3000)

print(f"MiniLM silhouette score : {minilm_similarity_score:}")
print(f"Specter silhouette score : {specter_similarity_score:}")

In [ ]:
# trying different k values

k_vals = []
for v in range(5,20):
    k_vals.append(v)

minilm_similarity_score_k = []
specter_similarity_score_k = []

for k in k_vals:
    k_means = KMeans(n_clusters=k,random_state=33)
    
    minilm_labels_k = k_means.fit_predict(embeddings)
    minilm_similarity_score_k.append(silhouette_score(embeddings,minilm_labels_k,metric='cosine',sample_size=3000))
    
    specter_labels_k = k_means.fit_predict(specter_model_embeddings)
    specter_similarity_score_k.append(silhouette_score(specter_model_embeddings,specter_labels_k,metric='cosine',sample_size=3000))
     
    print(f"k = {k}  MiniLM = {minilm_similarity_score_k[-1]:.4f}  Specter = {specter_similarity_score_k[-1]:.4f}")

In [ ]:
# plotting the silhouette scores

plt.figure(figsize=(10,6)) 
plt.plot(k_vals,minilm_similarity_score_k,marker='o',label='MiniLM') 
plt.plot(k_vals,specter_similarity_score_k,marker='o',label='Specter') 
plt.xlabel('K-values')
plt.ylabel('Silhouette score')  # higher the better
plt.title('Silhouette Score Comparison - MiniLM VS Specter')
plt.legend()
plt.grid(True,alpha=0.3)
plt.savefig('/content/silhouette_comparison_minilm_vs_specter.png',dpi=150)
plt.show()